In [0]:
%run ../config/config

In [0]:
import json
import pandas as pd
import numpy as np
import os
import sys
import time

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import NumericType
from datetime import datetime

sys.path.insert(0, str(project_path))

from utils.data_drift import DataDriftAnalyzer, DataDrift
from utils.logger import MLOpsLogger
from utils.metadata_serializer import MetadataSerializer

print("Imports completados exitosamente")

In [0]:
logger = MLOpsLogger(
    name='06_quality_outputs',
    log_level='DEBUG' if DEBUG_MLOPS else 'INFO',
    log_dir=logs_path,
    is_job_run=True,
    job_context={'mes_vta': mes_vta, 'environment': env, 'notebook': 'engine_drift'}
)

In [0]:
# =============================================================================
# CONFIGURACIÓN CENTRALIZADA
# =============================================================================
try:
    logger.info("=" * 60)
    logger.info("CARGANDO CONFIGURACIÓN PARA DRIFT")
    logger.info("=" * 60)

    from utils.config_loader import ConfigLoader
    config_loader = ConfigLoader(quality_config)

    column_settings = config_loader.get_column_settings()
    report_settings = config_loader.get_report_settings()
    drift_thresholds = config_loader.get_drift_thresholds()
    drift_metrics_to_consider = config_loader.get_drift_default_metrics()
    drift_bins = config_loader.get_drift_bins()

    campo_primario = column_settings.get('campo_primario', 'codclaveunicocli')
    columnas_control = column_settings.get('columnas_control', ['codclaveunicocli', 'codclavepartycli', 'codmes', 'fecproceso', 'fecdata'])
    titulo_drift_report = report_settings.get('titulo_drift', 'Reporte de Data Drift - Outputs')

    logger.info("✓ CONFIGURACIÓN DRIFT CARGADA")

except Exception as e:
    logger.log_exception(operation='load_drift_config', exception=e, should_raise=True)

In [0]:
# =============================================================================
# ★ CARGA DESDE DELTA STAGING (plan depth = 1)
# =============================================================================
try:
    logger.log_stage_start('engine_drift', mes_vta=mes_vta, environment=env)
    start_time = time.time()

    spark = SparkSession.builder.getOrCreate()

    # ★ Leer desde Delta staging - plan limpio, sin recursion
    persist_path = f"{adls_location_table}/quality_engine/{env}/{TABLE_OUTPUT}"

    logger.info(f"Leyendo desde Delta staging: {persist_path}")
    load_start = time.time()

    df_staging = spark.read.format("delta").load(persist_path)

    # ★ Filtrar actual e histórico - plan depth = 1
    df_actual = df_staging.filter(F.col("codmes") == mes_vta).cache()
    actual_count = df_actual.count()

    if actual_count == 0:
        raise ValueError(f"No hay registros para el mes {mes_vta}")

    df_historico = df_staging.filter(F.col("codmes") < mes_vta).cache()
    historico_count = df_historico.count()

    meses_historicos = sorted([r.codmes for r in df_historico.select('codmes').distinct().collect()])

    load_duration = time.time() - load_start
    logger.info(f"✓ Actual: {actual_count:,} | Histórico: {historico_count:,} ({len(meses_historicos)} meses) | {load_duration:.1f}s")

except Exception as e:
    logger.log_exception(operation='load_drift_data', exception=e, should_raise=True)

In [0]:
# =============================================================================
# CONFIGURACIÓN DE VARIABLES
# =============================================================================
try:
    todas_columnas = df_actual.columns
    columnas_numericas = [f.name for f in df_actual.schema.fields if isinstance(f.dataType, NumericType)]
    variables_existentes = [c for c in todas_columnas if c not in columnas_control and c in columnas_numericas]

    logger.info(f"Variables a analizar drift: {len(variables_existentes)} numéricas")

    if len(variables_existentes) == 0:
        raise ValueError("No hay variables numéricas para analizar drift")

except Exception as e:
    logger.log_exception(operation='configure_drift_variables', exception=e, should_raise=True)

In [0]:
# =============================================================================
# EJECUCION DE DRIFT (GLOBAL Y POR SEGMENTOS)
# =============================================================================
try:
    segments_to_run = {"global": (df_actual, df_historico)}

    if USE_SEGMENTATION:
        for seg_name in MODELS_CONFIG.keys():
            df_tgt_seg = df_actual.filter(F.col(SEGMENTATION_COLUMN) == seg_name)
            df_src_seg = df_historico.filter(F.col(SEGMENTATION_COLUMN) == seg_name)
            segments_to_run[seg_name] = (df_tgt_seg, df_src_seg)

    for segment_name, (df_tgt, df_src) in segments_to_run.items():
        logger.info("\n" + "=" * 80)
        logger.info(f"PROCESANDO DRIFT PARA SEGMENTO: {segment_name.upper()}")
        logger.info("=" * 80)

        if df_tgt.count() == 0 or df_src.count() == 0:
            logger.warning(f"Saltando segmento {segment_name}: registros insuficientes (Target: {df_tgt.count()}, Source: {df_src.count()})")
            continue

        # 1. Calcular Drift
        analyzer = DataDriftAnalyzer(df_base=df_src, df_actual=df_tgt, template_dir=None)
        drift_results_df = analyzer.run(columns=variables_existentes, drift_thresholds=drift_thresholds)
        drift_results_df = analyzer.add_summary_column(metrics_to_consider=drift_metrics_to_consider)
        semaphore_kpis = analyzer.calculate_semaphore_kpis()
        semaphore_config = analyzer.get_drift_semaphore_config()

        # 2. Data Match
        match_results = analyzer.comparator.get_variable_changes_summary(df_src.columns, df_tgt.columns)

        # 3. Guardar Metadatos
        if 'utils.metadata_serializer' in sys.modules:
            del sys.modules['utils.metadata_serializer']
        from utils.metadata_serializer import MetadataSerializer

        metadata_dir = os.path.join(temp_reports_input_path if 'inputs' in logger.name else temp_reports_output_path, "metadata", segment_name)
        os.makedirs(metadata_dir, exist_ok=True)
        serializer = MetadataSerializer(metadata_dir)

        bundle_path = serializer.save_metadata_bundle(
            mes_vta=mes_vta, environment=env, table_name=table_name_mt if 'inputs' in logger.name else output_table,
            total_records=df_src.count(), variables_analyzed=variables_existentes,
            current_metrics={}, evolutive_metrics={},
            drift_results=drift_results_df, variable_comparison=match_results,
            semaphore_kpis=semaphore_kpis, semaphore_config=semaphore_config, distribution_data_path=None,
            additional_metadata={}, filename="drift_metadata.json"
        )
        logger.info(f"✓ Drift guardado para {segment_name}: {bundle_path}")

    logger.log_stage_end('engine_drift', duration=time.time() - start_time)

except Exception as e:
    logger.log_exception(operation='engine_drift', exception=e, should_raise=True)